# Stage 2 Data Preparation: VietnamTourism Crawl + QA Generation + JSONL Export

This notebook crawls images from `vietnamtourism.gov.vn`, generates Vietnamese conversations with the OpenAI Batch API, then exports the result to Stage 2 instruction JSONL.

In [ ]:
!pip install -q -U gdown datasets pillow pandas pyarrow matplotlib seaborn tqdm pyyaml loguru beautifulsoup4 requests openai python-dotenv kaggle

import json
import os
import random
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

import yaml
try:
    from google.colab import userdata
except Exception:
    userdata = None

In [ ]:
REPO_URL = "https://github.com/duongtruongbinh/pretrain_vlm.git"
BRANCH = "heisenburger"
PROJECT_DIR = Path("/content/pretrain_vlm")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("Project root:", Path.cwd())

OPENAI_API_KEY = None
if userdata is not None:
    try:
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        OPENAI_API_KEY = None
OPENAI_API_KEY = OPENAI_API_KEY or os.environ.get("OPENAI_API_KEY")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("OPENAI_API_KEY is set.")
else:
    print("Set OPENAI_API_KEY in Colab Secrets or os.environ before running QA generation.")

## Configure a safe Colab run

The full run can take hours. Start with a small number of images, verify the output, then increase the limits.

In [ ]:
CRAWL_MAX_IMAGES = 50
CRAWL_MAX_PAGES = 3
QA_MAX_IMAGES = 20

config_path = Path("config.yaml")
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
config["crawl_vietnamtourism"]["max_images"] = CRAWL_MAX_IMAGES
config["crawl_vietnamtourism"]["max_pages"] = CRAWL_MAX_PAGES
config["generate_qa_vietnamtourism"]["max_images"] = QA_MAX_IMAGES
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
print("Updated config limits:", {"crawl_max_images": CRAWL_MAX_IMAGES, "crawl_max_pages": CRAWL_MAX_PAGES, "qa_max_images": QA_MAX_IMAGES})

## Crawl images and metadata

In [ ]:
subprocess.run(["python", "scripts/prepare_data.py", "vietnamtourism-crawl"], check=True)

## Visualize crawled records before QA generation

In [ ]:
def read_jsonl(path, limit=None):
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def plot_counts(title, counts):
    series = pd.Series(counts, dtype="int64")
    ax = series.plot(kind="bar", figsize=(7, 4), color="#4C78A8")
    ax.set_title(title)
    ax.set_ylabel("rows")
    ax.bar_label(ax.containers[0])
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def show_image_grid(records, text_getter, n=6):
    samples = [r for r in records if Path(str(r.get("image", ""))).exists()]
    if not samples:
        print("No local image samples found.")
        return
    samples = random.sample(samples, min(n, len(samples)))
    cols = min(3, len(samples))
    rows = (len(samples) + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 4.5 * rows))
    for i, record in enumerate(samples, start=1):
        image = Image.open(record["image"]).convert("RGB")
        ax = plt.subplot(rows, cols, i)
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(text_getter(record)[:120], fontsize=9)
    plt.tight_layout()
    plt.show()

raw_records = read_jsonl("data/vietnamtourism-raw/raw_crawl.jsonl", limit=200)
print("Raw crawled records:", len(raw_records))
display(pd.DataFrame(raw_records[:5])[['image_id', 'title', 'caption', 'article_url']])

for record in raw_records:
    record["image"] = record.get("image_path")
show_image_grid(raw_records, lambda r: f"{r.get('title', '')}\n{r.get('caption', '')}", n=6)

## Generate QA conversations with OpenAI Batch API

This step submits batch jobs and polls until completion. It may take minutes to hours depending on the number of images.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required for QA generation.")
subprocess.run(["python", "scripts/prepare_data.py", "vietnamtourism-qa"], check=True)

## Prepare final Stage 2 JSONL and visualize samples

In [ ]:
subprocess.run(["python", "scripts/prepare_data.py", "vietnamtourism-prepare"], check=True)

prepared_paths = {
    "train": "data/vietnamtourism/train.jsonl",
    "val": "data/vietnamtourism/val.jsonl",
    "test": "data/vietnamtourism/test.jsonl",
}
counts = {name: count_jsonl(path) for name, path in prepared_paths.items()}
display(pd.DataFrame([{"split": k, "rows": v} for k, v in counts.items()]))
plot_counts("VietnamTourism prepared rows", counts)

records = []
for path in prepared_paths.values():
    records.extend(read_jsonl(path, limit=100))
show_image_grid(records, lambda r: r.get("messages", [{}])[-1].get("content", ""), n=6)